# Algorithme EM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from random import randint,random

In [ ]:
def f(x):
  return 0.5*(1 - np.cos(np.pi*x))

def g(x):
  return f(f(x))

plt.plot(f(f(np.linspace(0,1,100))))
plt.show()

In [ ]:
N = 10000
F = 3
couleurs = ['red', 'green', 'blue', 'orange', 'purple', 'brown', 'pink', 'olive']

mu = [randint(0,30)+random() for i in range(F)]
sigma = [randint(1,4)+random() for i in range(F)]
nb = [randint(1,N//(F-1)) for i in range(F)]
nb[F-1] = N - (sum(nb) - nb[F-1])

echantillon = np.array([])
for i in range(F):
  echantillon = np.concat((echantillon, norm(mu[i], sigma[i]).rvs(nb[i])))
np.random.shuffle(echantillon)

n, _, _ = plt.hist(echantillon, bins = 100, color = 'grey')
x = np.linspace(min(echantillon), max(echantillon), 200)
for i in range(F):
    plt.plot(x, norm(mu[i],np.sqrt(sigma[i])).pdf(x)*nb[i]/3, color = couleurs[i])
plt.show()

In [ ]:
def EM_algorithm_step(pi, theta, x, n, F):
  new_rho = np.zeros((n,F))
  s = np.zeros(n)
  p = np.zeros((n,F))
  for i in range(n):
    for y in range(F):
      p[i,y] = 1/np.sqrt(2*np.pi*theta[y,1])*np.exp(-(x[i]-theta[y,0])**2/(2*theta[y,1]))
      s[i]+=pi[y]*p[i,y]
  for i in range(n):
    for y in range(F):
      new_rho[i,y] = pi[y]*p[i,y]/s[i]
  for y in range(F):
    new_pi=0
    for i in range(n):
      new_pi += new_rho[i,y]/n
    pi[y] = new_pi
  new_theta = np.zeros((F,2))
  for y in range(F):
    s1 = 0
    for i in range(n):
      s1 += new_rho[i,y]*x[i]
    new_theta[y,0] = s1/(n*pi[y])
    s2 = 0
    for i in range(n):
      s2 += new_rho[i,y]*(x[i] - new_theta[y,0])**2
    new_theta[y,1] = s2/(n*pi[y])
  return pi, new_theta

def EM_algorithm(n: int, F: int, x: np.array, pi0: np.array, theta0: np.array):
  pi, theta = pi0, theta0
  liste_pi, liste_theta = [np.copy(pi)], [np.copy(theta)]
  j=0
  while j<20:
    pi, theta = EM_algorithm_step(pi, theta, x, n, F)
    liste_pi.append(np.copy(pi))
    liste_theta.append(np.copy(theta))
    j += 1
  return pi, theta, liste_pi, liste_theta


In [ ]:
indices = np.arange(N)
np.random.shuffle(indices)
echantillon_shuffled = []
for i in indices:
  echantillon_shuffled.append(echantillon[i])
pi0 = np.zeros(F)
theta0 = np.zeros((F,2))
for y in range(F):
  pi0[y] = 1/F

for y in range(F):
  theta0[y,0] += np.mean(echantillon_shuffled[y*N//F : (y+1)*N//F])
  for i in range(y*N//F, (y+1)*N//F):
    theta0[y,1] += 1/N*(echantillon_shuffled[i]-theta0[y,0])**2
pi, theta, liste_pi, liste_theta = EM_algorithm(N, F, echantillon, pi0, theta0)
print( "pi = ", pi)
print(" theta = ", theta)

In [ ]:
pi0 = np.ones(F)/F
mu0 = np.linspace(min(echantillon), max(echantillon), F)
sigma0 = np.ones(F)*(max(echantillon)-min(echantillon))/F/F
theta0 = np.stack((mu0, sigma0), axis = -1)

pi, theta, liste_pi, liste_theta = EM_algorithm(N, F, echantillon, pi0, theta0)
print( "pi = ", pi)
print(" theta = ", theta)

In [ ]:
liste_theta = np.array(liste_theta)
liste_mu = np.split(liste_theta,2, axis = 2)[0]
liste_sigma = np.split(liste_theta,2, axis = 2)[1]

In [ ]:
def EM_algorithm_step_optim(pi, mu, sigma, x, n, F):
  p = np.zeros((F,n))
  if sigma.ndim <= 1:
    for y in range(F):
      p[y] = np.exp(-(x-mu[y])**2/(2*sigma[y]))/np.sqrt(2*np.pi*sigma[y])
  else:
    d = x[0].shape[0]
    for y in range(F):
      p[y] = np.exp(-np.einsum('ij, jk, ik -> i', (x-mu[y]), np.linalg.inv(sigma[y]), (x-mu[y]))/2)/np.sqrt((2*np.pi)**d*abs(np.linalg.det(sigma[y])))
  p = p.transpose()
  rho = (pi*p).transpose()/np.sum(pi*p, axis = 1)
  new_pi = np.mean(rho, axis = 1)
  new_mu = rho@x/np.sum(rho, axis = 1)
  new_sigma = np.empty_like(sigma)
  if sigma.ndim <= 1:
    for y in range(F):
      new_sigma[y] = np.sum(rho[y]*(x - new_mu[y])**2)/np.sum(rho[y])
  else:
    for y in range(F):
      new_sigma[y] = np.einsum('i, ij, ik-> jk',rho[y], (x-mu[y]), (x-mu[y]))/np.sum(rho[y])
  return new_pi, new_mu, new_sigma

def EM_algorithm_optim(pi0: np.array, mu0: np.array, sigma0: np.array, x: np.array, n: int, F: int):
  pi, mu, sigma = pi0, mu0, sigma0
  liste_pi, liste_mu, liste_sigma = [np.copy(pi0)], [np.copy(mu0)], [np.copy(sigma0)]
  j=0
  while j<200 and (len(liste_pi) <= 2
    or np.linalg.norm(liste_pi[-1] - liste_pi[-2])+np.linalg.norm(liste_mu[-1] - liste_mu[-2])+np.linalg.norm(liste_sigma[-1] - liste_sigma[-2]) > 0.001):
    pi, mu, sigma = EM_algorithm_step_optim(pi, mu, sigma, x, n, F)
    liste_pi.append(np.copy(pi))
    liste_mu.append(np.copy(mu))
    liste_sigma.append(np.copy(sigma))
    j += 1
  return pi, mu, sigma, liste_pi, liste_mu, liste_sigma

In [ ]:
rho = np.random.random((3,100))
x = np.random.random((100,10))
mu = np.random.random((3,10))

res = np.einsum('i, ij, ik-> jk',rho[0], (x-mu[0]), (x-mu[0]))
print(res.shape)
i, j, k = 70, 1, 3
print(np.sum(rho[0]*(x[:, j] - mu[0,j])*(x[:, k] - mu[0,k])))
print(res[j, k])

In [ ]:
indices = np.arange(N)
np.random.shuffle(indices)
echantillon_shuffled = []
for i in indices:
  echantillon_shuffled.append(echantillon[indices[i]])
pi0 = np.zeros(F)
theta0 = np.zeros((F,2))
for y in range(F):
  pi0[y] = 1/F

mu0 = np.zeros(F)
sigma0 = np.zeros(F)
for y in range(F):
  mu0[y] += np.mean(echantillon_shuffled[y*N//F : (y+1)*N//F])
  for i in range(y*N//F, (y+1)*N//F):
    sigma0[y] += 1/N*(echantillon_shuffled[i]-theta0[y,0])**2
_, _, _, liste_pi, liste_mu, liste_sigma = EM_algorithm_optim(pi0, mu0, sigma0, echantillon, N, F)

In [ ]:
pi0 = np.ones(F)/F
mu0 = np.linspace(min(echantillon), max(echantillon), F)
sigma0 = np.ones(F)*(max(echantillon)-min(echantillon))/F/F

_, _, _, liste_pi, liste_mu, liste_sigma = EM_algorithm_optim(pi0, mu0, sigma0, echantillon, N, F)

# Illustrations étapes

In [ ]:
fig, ax = plt.subplots()
n_frame = 180

n, x = np.histogram(echantillon, bins = 200)
x = x[:len(n)]

p = np.zeros((F,len(x)))
for y in range(F):
  p[y] = np.exp(-(x-mu0[y])**2/(2*sigma0[y]))/np.sqrt(2*np.pi*sigma0[y])
p = p.transpose()
rho = (pi0*p).transpose()/np.sum(pi0*p, axis = 1)

bottom = np.zeros(len(x))

ax.set_ylim([0,max(n)*1.1])
for y in range(F):
  ax.bar(x, height = max(n)*rho[y], bottom = bottom, color=couleurs[y])
  bottom += rho[y]*max(n)


def update(frame):
    ax.clear()
    t = f(frame/n_frame)
    bottom = np.zeros(len(x))
    ax.set_ylim([0,max(n)*1.1])
    for y in range(F):
      ax.bar(x, height = (max(n)*(1-t) + t*n)*rho[y], bottom = bottom, color=couleurs[y])
      bottom += rho[y]*(max(n)*(1-t) + t*n)

ani = animation.FuncAnimation(fig=fig, func=update, frames=n_frame, interval=10)
ani.save("animation_E1.gif")
HTML(ani.to_jshtml())

In [ ]:
fig, ax = plt.subplots(ncols = F)
n_frame = 180

n, x = np.histogram(echantillon, bins = 200)
x = x[:len(n)]

p = np.zeros((F,len(x)))
for y in range(F):
  p[y] = np.exp(-(x-mu0[y])**2/(2*sigma0[y]))/np.sqrt(2*np.pi*sigma0[y])
p = p.transpose()
rho = (pi0*p).transpose()/np.sum(pi0*p, axis = 1)

for y in range(F):
  ax[y].bar(x, height = max(n)*rho[y], color=couleurs[y])
  ax[y].set_ylim([0,max(n)*1.1])


def update(frame):
    t = f(frame/n_frame)
    for y in range(F):
      ax[y].clear()
      ax[y].bar(x, height = (max(n)*(1-t) + t*n)*rho[y], color=couleurs[y])
      ax[y].set_ylim([0,max(n)*1.1])

ani = animation.FuncAnimation(fig=fig, func=update, frames=n_frame, interval=10)
ani.save("animation_E2.gif")
HTML(ani.to_jshtml())

In [ ]:
fig, ax = plt.subplots()
n_frame = 180

n, x = np.histogram(echantillon, bins = 200)
x = x[:len(n)]

p = np.zeros((F,len(x)))
for y in range(F):
  p[y] = np.exp(-(x-mu0[y])**2/(2*sigma0[y]))/np.sqrt(2*np.pi*sigma0[y])
p = p.transpose()
rho = (pi0*p).transpose()/np.sum(pi0*p, axis = 1)

for y in range(F):
  ax.barh(x, width = max(n)*rho[y]*(-1)**y, color=couleurs[y])
  ax.set_xlim([-max(n)*1.1,max(n)*1.1])

def update(frame):
    ax.clear()
    t = f(frame/n_frame)
    for y in range(F):
      ax.barh(x, width = (max(n)*(1-t) + t*n)*rho[y]*(-1)**y, color=couleurs[y])
      ax.set_xlim([-max(n)*1.1,max(n)*1.1])

ani = animation.FuncAnimation(fig=fig, func=update, frames=n_frame, interval=10)
ani.save("animation_E3.gif")
HTML(ani.to_jshtml())

In [ ]:
fig, ax = plt.subplots()
n_frame = 180

x = np.linspace(min(echantillon), max(echantillon), 200)
gaussiennes = list()
nb,  _, hist = ax.hist(echantillon, bins = 100, color='grey')
for i in range(F):
  gaussiennes.append(ax.plot(x, norm(liste_mu[0][i],np.sqrt(liste_sigma[0][i])).pdf(x)*liste_pi[0][i]*N/2, color=couleurs[i])[0])
  plt.ylim([0, max(nb)*1.1])

def update(frame):
    t = g(frame/n_frame)
    pi = liste_pi[len(liste_pi)-1]*t + liste_pi[0]*(1-t)
    mu = liste_mu[len(liste_pi)-1]*t + liste_mu[0]*(1-t)
    sigma = liste_sigma[len(liste_pi)-1]*t + liste_sigma[0]*(1-t)
    for i in range(F):
        gaussiennes[i].set_ydata(norm(mu[i], np.sqrt(sigma[i])).pdf(x)*pi[i]*N/2)
    return gaussiennes

ani = animation.FuncAnimation(fig=fig, func=update, frames=n_frame, interval=10)
ani.save("animation_M1.gif")
HTML(ani.to_jshtml())

In [ ]:
fig, ax = plt.subplots()
n_frame = 180

x = np.linspace(min(echantillon), max(echantillon), 200)
gaussiennes = list()
nb,  _, hist = ax.hist(echantillon, bins = 100, color='grey')
for i in range(F):
  gaussiennes.append(ax.plot(x, norm(liste_mu[0][i],np.sqrt(liste_sigma[0][i])).pdf(x)*liste_pi[0][i]*N/2, color=couleurs[i])[0])
  plt.ylim([0, max(nb)*1.2])

def update(frame):
    t = g(frame%(n_frame//3)/n_frame * 3)
    if frame < n_frame//3:
      pi = liste_pi[len(liste_pi)-1]*t + liste_pi[0]*(1-t)
      mu = liste_mu[0]
      sigma = liste_sigma[0]
    elif frame < 2*n_frame//3:
      pi = liste_pi[len(liste_pi)-1]
      mu = liste_mu[len(liste_pi)-1]*t + liste_mu[0]*(1-t)
      sigma = liste_sigma[0]
    else:
      pi = liste_pi[len(liste_pi)-1]
      mu = liste_mu[len(liste_pi)-1]
      sigma = liste_sigma[len(liste_pi)-1]*t + liste_sigma[0]*(1-t)
    for i in range(F):
        gaussiennes[i].set_ydata(norm(mu[i], np.sqrt(sigma[i])).pdf(x)*pi[i]*N/2)
    return gaussiennes

ani = animation.FuncAnimation(fig=fig, func=update, frames=n_frame, interval=10)
ani.save("animation_M2.gif")
HTML(ani.to_jshtml())

# K-moyennes

In [ ]:
def nearest_neighbor(query, data):
    # Calculate L2 distances between the query vector and all data vectors
    if data.ndim > 1:
      distances = np.linalg.norm(data - query, axis = 1)
    else:
      distances = abs(data-query)

    # Find the index of the nearest neighbor
    nearest_index = np.argmin(distances)

    return nearest_index

def kmeans_step(centroid : np.array, x : np.array):
    nearest_index = np.array([nearest_neighbor(e, centroid) for e in x])
    return np.array([np.mean(x[nearest_index == i], axis = 0) for i in range(len(centroid))])


def kmeans(k, x):
  centroid = x[:k]*0
  np.random.shuffle(x)
  new_centroid = x[:k]
  i = 0
  while np.linalg.norm(centroid - new_centroid) > 0.001 and i < 100:
    i += 1
    centroid = new_centroid
    new_centroid = kmeans_step(centroid, x)
  return centroid


In [ ]:
pi0 = np.ones(F)/F
mu0 = kmeans(F, echantillon)
sigma0 = np.ones(F)*(max(echantillon)-min(echantillon))/F/F

_, _, _, liste_pi, liste_mu, liste_sigma = EM_algorithm_optim(pi0, mu0, sigma0, echantillon, N, F)

# Applications à d'autres lois

In [ ]:
from scipy.stats import binom, geom, poisson, uniform, expon, gamma, beta

In [ ]:
N = 10000
F = 3
couleurs = ['red', 'green', 'blue', 'orange', 'purple', 'brown', 'pink', 'olive']

n = [randint(2,50)*2 for i in range(F)]
p = [random() for i in range(F)]
nb = [randint(1,N//(F-1)) for i in range(F)]
nb[F-1] = N - (sum(nb) - nb[F-1])

echantillon = np.array([])
for i in range(F):
  echantillon = np.concat((echantillon, binom(n[i], p[i]).rvs(nb[i])))
np.random.shuffle(echantillon)

_, _, _ = plt.hist(echantillon, bins = 100, color = 'grey')
x = np.arange(min(echantillon), max(echantillon))
for i in range(F):
    plt.plot(x, binom(n[i],p[i]).pmf(x)*nb[i], color = couleurs[i])
plt.show()

In [ ]:
N = 10000
F = 3
couleurs = ['red', 'green', 'blue', 'orange', 'purple', 'brown', 'pink', 'olive']

p = [random() for i in range(F)]
nb = [randint(1,N//(F-1)) for i in range(F)]
nb[F-1] = N - (sum(nb) - nb[F-1])

echantillon = np.array([])
for i in range(F):
  echantillon = np.concat((echantillon, geom(p[i]).rvs(nb[i])))
np.random.shuffle(echantillon)

_, _, _ = plt.hist(echantillon, bins = 100, color = 'grey')
x = np.arange(min(echantillon), max(echantillon))
for i in range(F):
    plt.plot(x, geom(p[i]).pmf(x)*nb[i], color = couleurs[i])
plt.show()

In [ ]:
N = 10000
F = 3
couleurs = ['red', 'green', 'blue', 'orange', 'purple', 'brown', 'pink', 'olive']

p = [6/random() for i in range(F)]
nb = [randint(1,N//(F-1)) for i in range(F)]
nb[F-1] = N - (sum(nb) - nb[F-1])

echantillon = np.array([])
for i in range(F):
  echantillon = np.concat((echantillon, poisson(p[i]).rvs(nb[i])))
np.random.shuffle(echantillon)

_, _, _ = plt.hist(echantillon, bins = 100, color = 'grey')
x = np.arange(min(echantillon), max(echantillon))
for i in range(F):
    plt.plot(x, poisson(p[i]).pmf(x)*nb[i], color = couleurs[i])
plt.show()

In [ ]:
N = 10000
F = 3
couleurs = ['red', 'green', 'blue', 'orange', 'purple', 'brown', 'pink', 'olive']

n = [randint(2,50) for i in range(F)]
p = [randint(2,20)+random() for i in range(F)]
nb = [randint(1,N//(F-1)) for i in range(F)]
nb[F-1] = N - (sum(nb) - nb[F-1])

echantillon = np.array([])
for i in range(F):
  echantillon = np.concat((echantillon, uniform(n[i], p[i]).rvs(nb[i])))
np.random.shuffle(echantillon)

_, _, _ = plt.hist(echantillon, bins = 100, color = 'grey')
x = np.arange(min(echantillon), max(echantillon))
for i in range(F):
    plt.plot(x, uniform(n[i],p[i]).pdf(x)*nb[i], color = couleurs[i])
plt.show()

In [ ]:
N = 10000
F = 3
couleurs = ['red', 'green', 'blue', 'orange', 'purple', 'brown', 'pink', 'olive']

n = [randint(2,50)*2 for i in range(F)]
p = [randint(2,10)+random() for i in range(F)]
nb = [randint(1,N//(F-1)) for i in range(F)]
nb[F-1] = N - (sum(nb) - nb[F-1])

echantillon = np.array([])
for i in range(F):
  echantillon = np.concat((echantillon, expon(n[i], p[i]).rvs(nb[i])))
np.random.shuffle(echantillon)

_, _, _ = plt.hist(echantillon, bins = 100, color = 'grey')
x = np.arange(min(echantillon), max(echantillon))
for i in range(F):
    plt.plot(x, expon(n[i],p[i]).pdf(x)*nb[i], color = couleurs[i])
plt.show()

In [ ]:
N = 10000
F = 3
couleurs = ['red', 'green', 'blue', 'orange', 'purple', 'brown', 'pink', 'olive']


p = [8/random() for i in range(F)]
nb = [randint(1,N//(F-1)) for i in range(F)]
nb[F-1] = N - (sum(nb) - nb[F-1])

echantillon = np.array([])
for i in range(F):
  echantillon = np.concat((echantillon, gamma(p[i]).rvs(nb[i])))
np.random.shuffle(echantillon)

_, _, _ = plt.hist(echantillon, bins = 100, color = 'grey')
x = np.arange(min(echantillon), max(echantillon))
for i in range(F):
    plt.plot(x, gamma(p[i]).pdf(x)*nb[i], color = couleurs[i])
plt.show()

In [ ]:
N = 10000
F = 3
couleurs = ['red', 'green', 'blue', 'orange', 'purple', 'brown', 'pink', 'olive']

n = [uniform(0,10).rvs() for i in range(F)]
p = [uniform(0,10).rvs() for i in range(F)]
nb = [randint(1,N//(F-1)) for i in range(F)]
nb[F-1] = N - (sum(nb) - nb[F-1])

echantillon = np.array([])
for i in range(F):
  echantillon = np.concat((echantillon, beta(n[i], p[i]).rvs(nb[i])))
np.random.shuffle(echantillon)

_, _, _ = plt.hist(echantillon, bins = 100, color = 'grey')
x = np.linspace(min(echantillon), max(echantillon),200)
for i in range(F):
    plt.plot(x, beta(n[i],p[i]).pdf(x)*nb[i]/100, color = couleurs[i])
plt.show()

In [ ]:
pi0 = np.ones(F)/F
mu0 = kmeans(F, echantillon)
sigma0 = np.ones(F)*(max(echantillon)-min(echantillon))/F/F

pi1, mu1, sigma1, _, _, _ = EM_algorithm_optim(pi0, mu0, sigma0, echantillon, N, F)

n, _, _ = plt.hist(echantillon, bins = 100, color = 'grey')
x = np.linspace(min(echantillon), max(echantillon), 200)
for i in range(F):
    plt.plot(x, norm(mu1[i],np.sqrt(sigma1[i])).pdf(x)*pi1[i]*N/100, color = couleurs[i])
plt.show()

# Visualisation des résultats

In [ ]:
import ipywidgets as widgets  # interactive display
%config InlineBackend.figure_format = 'retina'

In [ ]:
@widgets.interact(etape=widgets.IntSlider(0, min=1, max=len(liste_pi),
                                              description="Etape"))

def progression(etape=0):
  x = np.linspace(min(echantillon), max(echantillon), 200)
  nb, _, _ = plt.hist(echantillon, bins = 100)
  for i in range(F):
      plt.plot(x, norm(liste_mu[etape-1][i],np.sqrt(liste_sigma[etape-1][i])).pdf(x)*liste_pi[etape-1][i]*N/2, color=couleurs[i])
  plt.show()

In [ ]:
import matplotlib.animation as animation
from IPython.display import HTML

In [ ]:
fig, ax = plt.subplots()

x = np.linspace(min(echantillon), max(echantillon), 200)
nb,  _, hist = ax.hist(echantillon, bins = 100)
gaussiennes = list()
for i in range(F):
  gaussiennes.append(ax.plot(x, norm(liste_mu[0][i],liste_sigma[0][i]).pdf(x)*liste_pi[0][i]*max(nb)*np.sqrt(2*np.pi*liste_sigma[0][i]), color=couleurs[i])[0])

def update(frame):
    pi = liste_pi[frame]
    mu = liste_mu[frame]
    sigma = liste_sigma[frame]
    for i in range(F):
        gaussiennes[i].set_ydata(norm(mu[i], np.sqrt(sigma[i])).pdf(x)*pi[i]*N/2)
    return gaussiennes

ani = animation.FuncAnimation(fig=fig, func=update, frames=len(liste_pi), interval=30)
ani.save("animation_I3.gif")
HTML(ani.to_jshtml())

In [ ]:
liste_mu = np.array(liste_mu)
liste_sigma = np.array(liste_sigma)
for i in range(F):
    plt.plot(liste_mu.transpose()[i], color = couleurs[i])
plt.show()
for i in range(F):
    plt.plot(liste_sigma.transpose()[i], color = couleurs[i])
plt.show()

# PCA

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_table("/content/WineQT.csv", sep=',')
data.drop("Id", axis = "columns", inplace = True)
data = (data - np.mean(data, axis = 0))/np.std(data, axis = 0)
data

In [ ]:
plt.hist(data["pH"], bins = 50)
plt.show()

In [ ]:
from sklearn import decomposition
pca = decomposition.PCA()
pca.fit(data)

In [ ]:
plt.plot(pca.explained_variance_)
plt.title("Quantité de variance expliqué par chaque composante")
plt.show()

n = len(pca.explained_variance_)
total = sum(pca.explained_variance_)
plt.plot([k for k in range(n)], [sum(pca.explained_variance_[:k])/total for k in range(n)])
plt.title("Proportion de la variance expliqué par chaque composante")
plt.show()

In [ ]:
train_scores = pca.transform(data) # Array of scores of the training set

plt.figure(figsize=(8, 8))

plt.scatter(train_scores[:,0],train_scores[:,1],s=0.5)
plt.show()

In [ ]:
loadings = pd.DataFrame(
    data=pca.components_.T * np.sqrt(pca.explained_variance_),
    columns=[f"PC{i}" for i in range(1, len(data.columns) + 1)],
    index=data.columns
)

fig, axs = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)
colors = ["#1C3041", "#9B1D20", "#0B6E4F", "#895884"]

for i, ax in enumerate(axs.flatten()):
    explained_variance = pca.explained_variance_ratio_[i] * 100
    pc = f"PC{i+1}"
    bars = ax.bar(loadings.index, loadings[pc], color=colors[i], edgecolor="#000000", linewidth=1.2)
    ax.set_title(f"{pc} Loading Scores ({explained_variance:.2f}% Explained Variance)", loc="left", fontdict={"weight": "bold"}, y=1.06)
    ax.set_xlabel("Feature")
    ax.set_ylabel("Loading Score")
    ax.grid(axis="y")
    ax.tick_params(axis="x", rotation=90)
    ax.set_ylim(-1, 1)

    for bar in bars:
        yval = bar.get_height()
        offset = yval + 0.02 if yval > 0 else yval - 0.15
        ax.text(bar.get_x() + bar.get_width() / 2, offset, f"{yval:.2f}", ha="center", va="bottom")


plt.show()

In [ ]:
labels = data.columns
n = len(labels)
coeff = np.transpose(pca.components_)
pc1 = pca.components_[:, 0]
pc2 = pca.components_[:, 1]

plt.figure(figsize=(8, 8))

for i in range(n):
    plt.arrow(x=0, y=0, dx=coeff[i, 0], dy=coeff[i, 1], color="#000000", width=0.003, head_width=0.03)
    plt.text(x=coeff[i, 0] * 1.15, y=coeff[i, 1] * 1.15, s=labels[i], size=13, color="#000000", ha="center", va="center")

plt.axis("square")
plt.title(f"Wine Quality Dataset PCA Biplot", loc="left", fontdict={"weight": "bold"}, y=1.06)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")

plt.xlim(-1, 1)
plt.ylim(-1, 1)
plt.xticks(np.arange(-1, 1.1, 0.2))
plt.yticks(np.arange(-1, 1.1, 0.2))

plt.axhline(y=0, color="black", linestyle="--")
plt.axvline(x=0, color="black", linestyle="--")
circle = plt.Circle((0, 0), 0.99, color="gray", fill=False)
plt.gca().add_artist(circle)

plt.grid()
plt.show()

# Clustering : k-moyennes

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

In [ ]:
k = 4
kmeans_model = KMeans(
    init='k-means++', # How to compute initial centroids
    max_iter=300, # Maximal number of iterations
    n_clusters=k, # Number of clusters
    n_init=20 # Number of times the algorithm is run with different initial configurations
)
kmeans = kmeans_model.fit(data)

labels = kmeans.labels_ # Array which contains the index of the cluster of each point
centroids = kmeans.cluster_centers_ # Array which contains the coordinates of the centroids

train_scores = pca.transform(data) # Array of scores of the training set
colors = plt.cm.tab10(np.linspace(0, 1, 10)) # Table of colors

plt.figure(figsize=(8, 8))

# Scatter plots
for y in range(k):
  indices = np.where((labels == y))
  train_scores_extracted = train_scores[indices]
  plt.scatter(train_scores_extracted[:,0],train_scores_extracted[:,1],color=colors[y])

plt.scatter(centroids[:, 0], centroids[:, 1], marker='*', c='black')
plt.show()

In [ ]:
wcss_kmeans = np.zeros(30)
silhouette_kmeans = np.zeros(30)

for k in range(1, 31):
  kmeans_model = KMeans(
    init='k-means++', # How to compute initial centroids
    max_iter=300, # Maximal number of iterations
    n_clusters=k, # Number of clusters
    n_init=20 # Number of times the algorithm is run with different initial configurations
  )
  kmeans = kmeans_model.fit(data)

  wcss_kmeans[k-1] = kmeans.inertia_
  if k == 1:
    silhouette_kmeans[0] = 0
  else:
    silhouette_kmeans[k-1] = silhouette_score(data, kmeans.labels_)

plt.plot(wcss_kmeans, 'b')
plt.title("WCSS K-means")
plt.show()
plt.plot(silhouette_kmeans, 'r')
plt.title("Silhouette score K-means")
plt.show()

# Clustering : Classification hiérarchique

In [ ]:
k = 6
agglo_ward_model = AgglomerativeClustering(linkage='ward', n_clusters=3)
agglo_ward = agglo_ward_model.fit(data)

labels = agglo_ward.labels_ # Array which contains the index of the cluster of each point

train_scores = pca.transform(data) # Array of scores of the training set
colors = plt.cm.tab10(np.linspace(0, 1, k)) # Table of colors

plt.figure(figsize=(8, 8))

# Scatter plots
for y in range(k):
  indices = np.where((labels == y))
  train_scores_extracted = train_scores[indices]
  plt.scatter(train_scores_extracted[:,0],train_scores_extracted[:,1],color=colors[y])

plt.show()

In [ ]:
def labels_matrix(children):
  n_s = children.shape[0]+1

  # clust[k,i] will contain the index of the cluster to which i belongs after k iterations
  clust = np.zeros((n_s,n_s), dtype=int)
  clust[0] = np.arange(n_s)

  # Browse children for agglomerative clustering
  for k in range(n_s-1):
    c1 = children[k,0] # index of 1st cluster merging
    c2 = children[k,1] # index of 2nd cluster merging
    for i in range(n_s):
      clust[k+1,i] = clust[k,i] # By default, clusters do not change indices
      if clust[k,i]==c1:
        clust[k+1,i] = n_s+k
      if clust[k,i]==c2:
        clust[k+1,i] = n_s+k

  return clust

In [ ]:
linkage = ["ward", "complete", "average", "single"]
for i in range(len(linkage)):
  agglo_model = AgglomerativeClustering(linkage=linkage[i], distance_threshold=0, n_clusters=None)
  agglo = agglo_model.fit(data)

  labels = labels_matrix(agglo.children_)
  silhouette_agglo = [silhouette_score(data, labels[-31+i]) for i in range(30)]
  silhouette_agglo.reverse()
  plt.plot(silhouette_agglo, label="agglo "+linkage[i])

plt.plot(silhouette_kmeans, 'b', label="k-means")
plt.legend()
plt.title("WCSS")
plt.show()

# Geyser

In [ ]:
import seaborn as sns
import pandas as pd

data = sns.load_dataset('geyser')
print(data.head())
plt.plot(data['duration'], data['waiting'], 'o')
plt.show()

In [ ]:
echantillon = data[['duration', 'waiting']].to_numpy(dtype = np.float128)
N = data.shape[0]
F = 2
pi0 = np.ones(F)/F
mu0 = kmeans(F, echantillon)
sigma0 = np.array([np.eye(echantillon.shape[1]) for _ in range(F)])/F/F

pi, mu, sigma, liste_pi, liste_mu, liste_sigma = EM_algorithm_optim(pi0, mu0, sigma0, echantillon, N, F)

In [ ]:

plt.scatter(data.loc[data['kind']=='long','duration'], data.loc[data['kind']=='long', 'waiting'], color='olive')
plt.scatter(data.loc[data['kind']=='short','duration'], data.loc[data['kind']=='short','waiting'], color='brown')
plt.scatter(mu0[:,0], mu0[:, 1])

In [ ]:
from matplotlib.patches import Ellipse

plt.figure(figsize=(8, 8))
plt.scatter(data.loc[data['kind']=='long','duration'], data.loc[data['kind']=='long', 'waiting'], color='olive', label='long')
plt.scatter(data.loc[data['kind']=='short','duration'], data.loc[data['kind']=='short','waiting'], color='brown', label='short')
plt.legend()

# Adding centres of mass and ellipses
for y in range(F):
  # Centre of mass
  plt.scatter(mu[y,0], mu[y, 1], color=couleurs[y])

  # Ellipse
  # Construct the ellipse
  eigenvalues, eigenvectors = np.linalg.eigh(sigma[y,0:2,0:2]) # Eigenvalues and eigenvectors of the covariance matrix
  angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))  # Rotation angle
  width, height = 2 * np.sqrt(6*eigenvalues) # Width and height of ellipse
   # (scaled by 6 which is the quantile of order 95% of the chi2 distribution with 2 dof)

  # Create ellipse patch
  ellipse = Ellipse(xy=mu[y,0:2], width=width, height=height, angle=angle,
                      edgecolor=couleurs[y], facecolor='none', linewidth=2)

  ax = plt.gca()
  ax.add_patch(ellipse)

plt.show()